In [1]:
import os
# Suppress TensorFlow info and warning logs for a cleaner terminal
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Set random seeds for reproducibility (helps match lab sheet results)
tf.random.set_seed(42)
np.random.seed(42)

def build_optimized_mlp(input_dim, hidden_units=128, dropout_rate=0.3, lr=0.1):
    """Builds the MLP with configurable width and dropout for the Iris dataset."""
    model = models.Sequential([
        layers.Dense(hidden_units, activation='relu', input_shape=(input_dim,)),
        layers.Dropout(dropout_rate),
        # Output layer uses 3 units and softmax because Iris has 3 distinct classes
        layers.Dense(3, activation='softmax')
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

if __name__ == "__main__":
    print("--- Lab 3: Advanced Hyperparameter Optimization ---")

    try:
        # 1. Get Hyperparameters
        lr_input = input("Enter learning rate (press Enter for default 0.1): ")
        lr = float(lr_input) if lr_input.strip() else 0.1

        units_input = input("Enter number of hidden units (press Enter for default 128): ")
        hidden_units = int(units_input) if units_input.strip() else 128

        batch_input = input("Enter batch size (press Enter for default 64): ")
        batch_size = int(batch_input) if batch_input.strip() else 64

        dropout_input = input("Enter dropout rate (press Enter for default 0.3): ")
        dropout_rate = float(dropout_input) if dropout_input.strip() else 0.3

        epochs = 100 # Fixed at 100 per the Lab 3 specifications

        # 2. Load and Preprocess Dataset
        print("\nLoading and preprocessing Iris dataset (80/20 split)...")
        iris = load_iris()
        X, y = iris.data, iris.target

        # 80/20 Train/Validation Split
        X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

        # Standardize features (crucial for stable DNN training)
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_val = scaler.transform(X_val)

        # 3. Build and Train Model
        print(f"\nBuilding MLP (Input -> Dense({hidden_units}, relu) -> Dropout({dropout_rate}) -> Dense(3, softmax))...")
        model = build_optimized_mlp(input_dim=X_train.shape[1], hidden_units=hidden_units, dropout_rate=dropout_rate, lr=lr)

        print(f"Training for {epochs} epochs with batch size {batch_size}...")
        history = model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size,
                            validation_data=(X_val, y_val), verbose=0)

        # 4. Output Final Results
        final_train_loss = history.history['loss'][-1]
        final_val_acc = history.history['val_accuracy'][-1] * 100

        print("\n--- Final Results ---")
        print(f"Configuration : lr={lr}, units={hidden_units}, batch={batch_size}, dropout={dropout_rate}")
        print(f"Train Loss    : {final_train_loss:.4f}")
        print(f"Val Accuracy  : {final_val_acc:.1f}%")

        # Simple Overfitting Check (if training loss is very low but val accuracy is poor)
        if final_train_loss < 0.1 and final_val_acc < 80.0:
            print("Status        : Overfitting detected (High capacity, poor generalization).")
        else:
            print("Status        : Model generalized well.")

    except ValueError as e:
        print(f"\nInput Error: {e}")
        print("Please restart the script and enter valid numbers.")

--- Lab 3: Advanced Hyperparameter Optimization ---
Enter learning rate (press Enter for default 0.1): 
Enter number of hidden units (press Enter for default 128): 
Enter batch size (press Enter for default 64): 
Enter dropout rate (press Enter for default 0.3): 

Loading and preprocessing Iris dataset (80/20 split)...

Building MLP (Input -> Dense(128, relu) -> Dropout(0.3) -> Dense(3, softmax))...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Training for 100 epochs with batch size 64...

--- Final Results ---
Configuration : lr=0.1, units=128, batch=64, dropout=0.3
Train Loss    : 0.0354
Val Accuracy  : 100.0%
Status        : Model generalized well.
